# Linked Nodes

```{contents}
:local:
:depth: 2
```

A linked list stores a sequence as separate node objects connected by references. The representation makes insertion and deletion local, but finding a position requires traversal.


```{index} node
```
```{index} linked list
```

## Nodes and References


Each node stores a value and a reference to the next node.

```text
head -> [10 | next] -> [20 | next] -> [30 | null]
```

The list variable stores a reference to the first node, often called the **head**. The last node points to `null`.


In [ ]:
IntroNode<int> first = new IntroNode<int>(10);
first.Next = new IntroNode<int>(20);
first.Next.Next = new IntroNode<int>(30);

Console.WriteLine(first.Value);
Console.WriteLine(first.Next.Value);

public class IntroNode<T>
{
    public T Value { get; }
    public IntroNode<T>? Next { get; set; }

    public IntroNode(T value)
    {
        Value = value;
    }
}

```{index} traversal; linked list
```

## Traversal


A linked list cannot jump directly to index `7`. To find an item, code follows references one node at a time. This is called **traversal**.


In [ ]:
TraversalNode<string> head = new TraversalNode<string>("alpha");
head.Next = new TraversalNode<string>("beta");
head.Next.Next = new TraversalNode<string>("gamma");

int visits = 0;
for (TraversalNode<string>? current = head; current is not null; current = current.Next)
{
    visits++;
    Console.WriteLine($"visit {visits}: {current.Value}");
}

public class TraversalNode<T>
{
    public T Value { get; }
    public TraversalNode<T>? Next { get; set; }
    public TraversalNode(T value) => Value = value;
}

```{index} insertion; linked list
```

## Insertion After a Node


If you already have a reference to a node, inserting after it is local. Create a new node, point it to the old next node, then point the current node to the new node.

```text
before: [A] -> [C]
after:  [A] -> [B] -> [C]
```


In [ ]:
static void InsertAfter<T>(InsertNode<T> node, T value)
{
    InsertNode<T> inserted = new InsertNode<T>(value);
    inserted.Next = node.Next;
    node.Next = inserted;
}

InsertNode<string> head = new InsertNode<string>("A");
head.Next = new InsertNode<string>("C");
InsertAfter(head, "B");

for (InsertNode<string>? current = head; current is not null; current = current.Next)
{
    Console.WriteLine(current.Value);
}

public class InsertNode<T>
{
    public T Value { get; }
    public InsertNode<T>? Next { get; set; }
    public InsertNode(T value) => Value = value;
}

```{index} deletion; linked list
```

## Deletion After a Node


Deleting after a known node is also local. Skip over the next node by changing one reference.

```text
before: [A] -> [B] -> [C]
after:  [A] --------> [C]
```


In [ ]:
static bool DeleteAfter<T>(DeleteNode<T> node)
{
    if (node.Next is null) return false;
    node.Next = node.Next.Next;
    return true;
}

DeleteNode<string> head = new DeleteNode<string>("A");
head.Next = new DeleteNode<string>("B");
head.Next.Next = new DeleteNode<string>("C");

DeleteAfter(head);

for (DeleteNode<string>? current = head; current is not null; current = current.Next)
{
    Console.WriteLine(current.Value);
}

public class DeleteNode<T>
{
    public T Value { get; }
    public DeleteNode<T>? Next { get; set; }
    public DeleteNode(T value) => Value = value;
}

```{index} Costs and Tradeoffs
```

## Costs and Tradeoffs


Linked lists make some updates cheap only after the position is known.

| Operation | Singly linked list cost |
| --- | --- |
| Read first item | O(1) |
| Traverse all items | O(n) |
| Find by value | O(n) |
| Insert after known node | O(1) |
| Delete after known node | O(1) |
| Read by index | O(n) |

The phrase "known node" matters. If the program must search for the node first, that search is part of the total cost.


```{index} alias; linked list; reference
```

## References Are Paths, Not Copies

Work orders may also be represented by separate node objects. A node's `Next` is a reference to another node, not an embedded copy of the remaining list. Assigning `current = head` copies a reference: both variables initially identify the same object. Changing `current` to `current.Next` moves a cursor without moving `head`.

```text
head ----> [WO-10 | next] ---> [WO-30 | null]
current --^
```

If you traverse by repeatedly assigning to `head`, you lose your entry point to the sequence. A separate cursor preserves the list. Also distinguish changing a reference from changing an object: setting `current = current.Next` moves the cursor; setting `current.Next = ...` changes a link in the sequence.

**Trace (basic):** Draw both arrows before and after one cursor advance. Which variable still reaches the first work order?

```{index} linked list; insertion invariant; cycle
```

## Insertion Order and Lost Links

You know the node for `WO-10` and want to insert `WO-20` before `WO-30`. Preserve the old successor before overwriting its only incoming list link.

| Step | References | What remains reachable |
| --- | --- | --- |
| Create B | `A.Next = C`, `B.Next = null` | A still reaches C. |
| Save successor | `B.Next = A.Next` | B now reaches C too. |
| Connect B | `A.Next = B` | A reaches B, then C. |

The correct two assignments are `inserted.Next = node.Next; node.Next = inserted;`. Reversing them makes `inserted.Next` refer to `inserted` itself. The result is a cycle, and the original successor is disconnected from this chain. A traversal waiting for `null` would never finish.

For an empty list there is no predecessor. Inserting at the front sets `newNode.Next = head` before replacing `head`. The same steps work when the old head is null. A list wrapper is useful because it owns the head reference and can update it safely.

**Debug (applied):** Explain the reversed-assignment bug using named nodes A, B, and C. Do not run an unbounded traversal of the broken chain.

```{index} linked list; removal; head; tail
```

## Removal, Head, and Tail Invariants

To remove B after A, preserve B's successor by setting `A.Next = B.Next`. Removing the first node instead requires `head = head.Next`, because there is no preceding node to update. Unlinking does not destroy the object immediately: other references might still reach it, and .NET manages reclamation of unreachable objects.

A wrapper with a tail reference must maintain these rules:

- An empty list has count zero and both head and tail null.
- A nonempty list has a reachable head and tail, and `tail.Next` is null.
- Following `Next` from head visits exactly `Count` nodes and reaches tail last.

Removing the only node must clear both endpoints. Removing the last node of a longer chain must move tail to the predecessor. A tail reference makes append constant-time but does not make removal of the last node constant-time: a singly linked list still has to find that predecessor.

**Predict (applied):** A one-node list is emptied and then appended to. What goes wrong if removal leaves tail pointing to the old node?

```{index} linked list; generic implementation; tail reference
```

## A Complete Singly Linked Sequence

This wrapper presents the same `Add`, `Insert`, indexer, and `RemoveAt` operations as the dynamic array. Nodes are private so client code cannot accidentally create cycles or change count without updating the wrapper.

`NodeAt` validates the index, then follows links. `head!` and `Next!` express facts established by the list invariants after validation; they do not create runtime null checks. If those invariants are broken, the suppression cannot repair them.

Run the cell independently. It inserts the middle order, removes the head, empties the list, and appends again. The last line should be `count=1, first=WO-40`.

In [ ]:
using System;
var orders = new WorkOrderChain<string>();
orders.Add("WO-10");
orders.Add("WO-30");
orders.Insert(1, "WO-20");
for (int i = 0; i < orders.Count; i++) Console.WriteLine(orders[i]);
Console.WriteLine($"removed={orders.RemoveAt(0)}");
orders.RemoveAt(1);
orders.RemoveAt(0);
orders.Add("WO-40");
Console.WriteLine($"count={orders.Count}, first={orders[0]}");
public class WorkOrderChain<T>
{
    private sealed class Node
    {
        public T Value;
        public Node? Next;
        public Node(T value, Node? next = null) { Value = value; Next = next; }
    }
    private Node? head;
    private Node? tail;
    public int Count { get; private set; }
    public T this[int index] => NodeAt(index).Value;
    private Node NodeAt(int index)
    {
        if (index < 0 || index >= Count)
            throw new ArgumentOutOfRangeException(nameof(index));
        Node current = head!;
        for (int i = 0; i < index; i++) current = current.Next!;
        return current;
    }
    public void Add(T value)
    {
        var added = new Node(value);
        if (tail is null) head = added;
        else tail.Next = added;
        tail = added;
        Count++;
    }
    public void Insert(int index, T value)
    {
        if (index < 0 || index > Count)
            throw new ArgumentOutOfRangeException(nameof(index));
        if (index == Count) { Add(value); return; }
        if (index == 0) head = new Node(value, head);
        else
        {
            Node previous = NodeAt(index - 1);
            previous.Next = new Node(value, previous.Next);
        }
        Count++;
    }
    public T RemoveAt(int index)
    {
        if (index < 0 || index >= Count)
            throw new ArgumentOutOfRangeException(nameof(index));
        Node removed;
        if (index == 0)
        {
            removed = head!;
            head = removed.Next;
            if (head is null) tail = null;
        }
        else
        {
            Node previous = NodeAt(index - 1);
            removed = previous.Next!;
            previous.Next = removed.Next;
            if (ReferenceEquals(removed, tail)) tail = previous;
        }
        Count--;
        return removed.Value;
    }
}

Notice that the interface hides implementation choices. `Insert(1, value)` first locates the predecessor and then rewires links. Although the rewiring is O(1), this index-based operation is O(n) in the worst case. The earlier `InsertAfter` example had a different assumption: its caller already held a node reference.

The demonstration prints by repeated indexing only because the list is tiny. For a long linked list, that pattern repeats traversal: indexes zero through `n - 1` require roughly `0 + 1 + ... + (n - 1)` link advances. A cursor or iterator should visit the sequence once instead.

**Extend (challenge):** Add an iterator using chapter 14's `yield return`. Compare a single traversal with repeated indexer reads. Explain what should happen if clients modify the list during iteration; production collections define such behavior explicitly.

```{index} doubly linked list; predecessor; memory overhead
```

## Doubly Linked Lists: A Different Tradeoff

A doubly linked node stores both `Next` and `Previous`. A known node can therefore find its predecessor without a search, and a stored tail can support constant-time removal at the end. The price is another reference per node and more links to maintain.

```text
null <- [A] <-> [B] <-> [C] -> null
```

Removing B connects `A.Next` to C and `C.Previous` to A. Removing an endpoint also changes the wrapper's head or tail. This representation still does not provide direct indexing: to find a numeric position, follow links from an endpoint.

**Compare (applied):** For frequent removal of a known item, explain what information singly and doubly linked lists require. Include the cost of locating the item if it is supplied only by value.

```{index} practice
```

## Practice


1. Draw the references after inserting `B` between `A` and `C`.
2. Explain why linked-list indexed access is O(n).
3. Write a method that counts nodes in a linked list starting from `head`.


```{index} summary
```

## Summary


A linked list stores sequence order through references rather than contiguous slots. This makes local insertion and deletion simple, but traversal is unavoidable for search and indexed access.


```{rubric} Footnotes
```
[^1]: Later chapters use linked nodes again for stacks, queues, trees, and graphs.
